# wz-agent 交互教程（v2.0）

## 项目简介

wz-agent 是一个从零手搓的通用编码助手，基于 DeepSeek API，采用 ReAct（Reasoning + Acting）范式。
本 Notebook 带你逐步体验核心流程：API 调用 → ReAct 循环 → 工具调用 → 分诊 / 任务拆解。

> ⚠️ 本 Notebook 是**教学简化版**：自带的 `CodingAgent` 类用于讲解原理。
> 最新完整实现见 `src/agent/`（schema 反射、bash 安全模式、自动重试、triage/to-tickets 均已落地）。

## 版本演进

- **v0.1 ~ v0.2**：API 连通、ReAct 循环、read/write/edit/bash 四个基础工具
- **v0.3 ~ v1.0**：bash 安全模式（auto/plan）、需求澄清 → 编码执行两阶段、click CLI、自动重试
- **v2.0**：triage（issue 分诊状态机）+ to-tickets（垂直切片任务拆解）

## 作者信息

- 姓名: Wz9899
- GitHub: @Wz9899
- 更新日期: 2026-08-01（同步至 v2.0）

# ========================================
# 第1部分：环境配置
# ========================================

In [ ]:
# 安装依赖（如已安装可跳过）
!pip install -q openai python-dotenv click rich

In [ ]:
# 导入必要的库
import os
import sys
from dotenv import load_dotenv
from openai import OpenAI
from rich.console import Console
from rich.panel import Panel

# 加载环境变量
load_dotenv()

console = Console()

# 检查 API Key
api_key = os.environ.get("DEEPSEEK_API_KEY")
if api_key:
    console.print("[green]✅ DEEPSEEK_API_KEY 已设置[/]")
else:
    console.print("[red]❌ 请设置 DEEPSEEK_API_KEY 环境变量[/]")
    console.print("  export DEEPSEEK_API_KEY='sk-你的key'")

# ========================================
# 第2部分：工具定义（4 个基础工具）
# ========================================

先手写工具类的 schema（教学用，直观展示 OpenAI Function Calling 协议）。
v1.0 之后项目改为**反射自动生成** schema —— 见 `src/agent/tools/base.py`。

In [ ]:
import json
from typing import Any


class ReadTool:
    """读取文件内容的工具"""

    name = "read"
    description = "读取指定路径的文件内容"

    def to_openai_function(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "文件路径（相对或绝对）"}
                    },
                    "required": ["path"]
                }
            }
        }

    def run(self, path: str) -> str:
        try:
            with open(path, "r", encoding="utf-8") as f:
                content = f.read()
            if len(content) > 3000:
                content = content[:3000] + "\n... (内容已截断)"
            return content
        except FileNotFoundError:
            return f"错误：文件不存在 —— {path}"
        except Exception as e:
            return f"错误：{e}"


class WriteTool:
    """创建或覆盖文件的工具"""

    name = "write"
    description = "创建新文件或覆盖已有文件，会自动创建父目录"

    def to_openai_function(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "文件路径"},
                        "content": {"type": "string", "description": "要写入的内容"}
                    },
                    "required": ["path", "content"]
                }
            }
        }

    def run(self, path: str, content: str) -> str:
        try:
            os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
            with open(path, "w", encoding="utf-8") as f:
                f.write(content)
            return f"✅ 文件已写入：{path}（{len(content)} 字符）"
        except Exception as e:
            return f"错误：{e}"


console.print("[green]✅ ReadTool / WriteTool 已定义[/]")

In [ ]:
class EditTool:
    """编辑文件：精确文本匹配替换（v0.2 新增）

    安全约束（与 src/agent/tools/edit.py 一致）：
        1. oldText 必须在文件中恰好出现 1 次，否则拒绝
        2. oldText 和 newText 不能相同
    """

    name = "edit"
    description = "在文件中精确查找一段文本并替换为新文本，oldText 必须唯一"

    def to_openai_function(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "要编辑的文件路径"},
                        "oldText": {"type": "string", "description": "被替换的原始文本（必须精确匹配且唯一）"},
                        "newText": {"type": "string", "description": "替换后的新文本"}
                    },
                    "required": ["path", "oldText", "newText"]
                }
            }
        }

    def run(self, path: str, oldText: str, newText: str) -> str:
        if oldText == newText:
            return "错误：oldText 和 newText 完全相同，无需替换"
        try:
            with open(path, "r", encoding="utf-8") as f:
                original = f.read()
        except FileNotFoundError:
            return f"错误：文件不存在 —— {path}"

        count = original.count(oldText)
        if count == 0:
            return "错误：未找到 oldText（注意空格和换行要完全一致）"
        if count > 1:
            return f"错误：oldText 出现了 {count} 次，必须唯一，请提供更长的上下文"

        line_no = original[: original.index(oldText)].count("\n") + 1
        with open(path, "w", encoding="utf-8") as f:
            f.write(original.replace(oldText, newText, 1))
        return f"✅ 已在 {path} 第 {line_no} 行完成替换"


console.print("[green]✅ EditTool 已定义（v0.2）[/]")

In [ ]:
class BashTool:
    """执行 Shell 命令（v0.3+：auto/plan 双模式 + 危险命令拦截）

    - auto: 直接执行（默认，低风险操作）
    - plan: 命令先收集到计划，调用 __execute_plan__ 时批量执行（高风险操作）
    """

    name = "bash"
    description = "执行 shell 命令并返回输出"

    TIMEOUT = 30
    DANGEROUS = ("rm -rf /", "rm -rf ~", "dd if=", "mkfs.", ":(){ :|:& };:", "> /dev/sda")

    def __init__(self):
        self.mode = "auto"
        self._plan = []

    def to_openai_function(self) -> dict:
        desc = self.description
        if self.mode == "plan":
            desc += " [WARN] 当前为计划模式：命令不立即执行，收集到计划；" \
                     "完成后调用 command='__execute_plan__' 批量执行"
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": desc,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "command": {"type": "string", "description": "要执行的 shell 命令"},
                        "reason": {"type": "string", "description": "命令用途说明（plan 模式用）"}
                    },
                    "required": ["command"]
                }
            }
        }

    def run(self, command: str, reason: str = "") -> str:
        cmd = command.strip()
        if self.mode == "plan":
            if cmd == "__show_plan__":
                return self._format_plan()
            if cmd == "__execute_plan__":
                return self._execute_plan()
            if cmd == "__clear_plan__":
                self._plan.clear()
                return "[CLEAR] 已清空执行计划"
            self._plan.append(cmd)
            reason_str = f" —— {reason}" if reason else ""
            return f"[PLAN] 已记录到计划 #{len(self._plan)}: `{cmd}`{reason_str}"
        return self._execute(cmd)

    def _format_plan(self) -> str:
        if not self._plan:
            return "[PLAN] 执行计划为空"
        lines = [f"[PLAN] 执行计划（共 {len(self._plan)} 条）:"]
        for i, c in enumerate(self._plan, 1):
            lines.append(f"  [{i}] `{c}`")
        return "\n".join(lines)

    def _execute_plan(self) -> str:
        if not self._plan:
            return "[PLAN] 执行计划为空"
        results = [f"[EXEC] 开始执行计划（共 {len(self._plan)} 条）..."]
        ok = fail = 0
        for c in self._plan:
            out = self._execute(c)
            results.append(f"── `{c}` ──\n{out}")
            if out.startswith("[ERR]"):
                fail += 1
            else:
                ok += 1
        self._plan.clear()
        results.append(f"[DONE] 计划执行完毕：{ok} 成功, {fail} 失败")
        return "\n".join(results)

    def _execute(self, command: str) -> str:
        import subprocess
        for d in self.DANGEROUS:
            if d in command:
                return f"[ERR] 安全拦截：命令包含 '{d}'"
        try:
            result = subprocess.run(
                command, shell=True, capture_output=True, text=True, timeout=self.TIMEOUT
            )
        except subprocess.TimeoutExpired:
            return f"[ERR] 超时（>{self.TIMEOUT}s），命令已被终止"
        except Exception as e:
            return f"[ERR] 执行出错：{e}"
        parts = []
        if result.stdout.strip():
            parts.append(result.stdout.rstrip())
        if result.stderr.strip():
            parts.append("[stderr]\n" + result.stderr.rstrip())
        return "\n".join(parts) if parts else "（命令执行完毕，无输出）"


console.print("[green]✅ BashTool 已定义（v0.3：auto/plan 双模式）[/]")

In [ ]:
# 工具注册表
ALL_TOOLS: dict[str, Any] = {
    "read": ReadTool(),
    "write": WriteTool(),
    "edit": EditTool(),
    "bash": BashTool(),
}

console.print("[green]✅ 工具已注册:[/]", ", ".join(ALL_TOOLS.keys()))

# ========================================
# 第3部分：智能体构建（ReAct + 自动重试）
# ========================================

`CodingAgent` 是教学版 ReAct 引擎（对应 `src/agent/loop.py`）：
- `run()`：思考 → 行动 → 观察 循环，直到 LLM 给出最终回复
- `run_with_retry()`（v1.0）：结果以 [ERR]/[WARN] 开头时，把失败详情回喂 LLM 自动修复，最多 3 次

In [ ]:
DEFAULT_SYSTEM_PROMPT = """你是一个编码助手，可以用中文回答。
你有以下工具可用：
- read: 读取文件内容
- write: 创建或覆盖文件
- edit: 精确文本匹配替换（oldText 必须唯一）
- bash: 执行 shell 命令（当前为 {mode} 模式）

当需要操作文件或执行命令时，请使用工具。完成任务后，总结你做了什么。"""


class CodingAgent:
    """ReAct 编码智能体（v2.0 教学版）"""

    def __init__(self, api_key: str, model: str = "deepseek-chat",
                 bash_mode: str = "auto", system_prompt: str | None = None):
        self.client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")
        self.model = model
        self.bash_mode = bash_mode
        self.system_prompt = system_prompt or DEFAULT_SYSTEM_PROMPT.format(mode=bash_mode)
        # 切换 bash 安全模式（影响 bash 工具的 description，LLM 会看到提示）
        ALL_TOOLS["bash"].mode = bash_mode

    def _run_messages(self, messages: list, max_steps: int = 10) -> str:
        """单轮 ReAct 循环核心：给定消息历史，循环直到任务完成。"""
        tool_schemas = [t.to_openai_function() for t in ALL_TOOLS.values()]

        for step in range(max_steps):
            console.print(f"\n[bold yellow]--- 第 {step + 1} 步 ---[/]")

            # 调用 LLM（带工具 schema）
            try:
                response = self.client.chat.completions.create(
                    model=self.model, messages=messages, tools=tool_schemas
                )
            except Exception as e:
                return f"[ERR] API 调用失败：{e}"

            msg = response.choices[0].message

            if msg.tool_calls:
                # 把 assistant 消息（含 tool_calls）加入对话
                messages.append(msg.model_dump(exclude_none=True))

                for tool_call in msg.tool_calls:
                    tool_name = tool_call.function.name
                    tool_args = json.loads(tool_call.function.arguments)
                    console.print(f"  🔧 调用工具: [cyan]{tool_name}[/] {tool_args}")

                    # 执行工具（包裹 try-catch，防止工具异常导致循环崩溃）
                    tool = ALL_TOOLS.get(tool_name)
                    try:
                        result = tool.run(**tool_args) if tool else f"未知工具：{tool_name}"
                    except Exception as e:
                        result = f"错误：工具 '{tool_name}' 执行异常 —— {e}"

                    result_display = result[:300] + "..." if len(result) > 300 else result
                    console.print(f"  📋 结果: {result_display}")

                    # 把工具结果加入对话
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": result,
                    })
            else:
                # LLM 直接回复 → 任务完成
                final_reply = msg.content or ""
                console.print(f"\n[bold green]✅ Agent 完成：[/]\n{final_reply}")
                return final_reply

        return "[WARN] 达到最大步数限制，Agent 未完成任务。"

    def run(self, user_message: str, system_prompt: str | None = None, max_steps: int = 10) -> str:
        """ReAct 循环入口：思考 → 行动 → 观察 → 再思考。"""
        prompt = system_prompt or self.system_prompt
        messages = [
            {"role": "system", "content": prompt},
            {"role": "user", "content": user_message},
        ]
        return self._run_messages(messages, max_steps)

    def run_with_retry(self, user_message: str, system_prompt: str | None = None,
                       max_retries: int = 3, max_steps: int = 10) -> str:
        """v1.0：失败自动修复 —— [ERR]/[WARN] 开头时把失败详情回喂 LLM，最多 3 次。"""
        prompt = system_prompt or self.system_prompt
        messages = [{"role": "system", "content": prompt}]
        result = ""
        for attempt in range(max_retries + 1):
            if attempt == 0:
                msgs = messages + [{"role": "user", "content": user_message}]
            else:
                msgs = messages + [{"role": "user", "content": (
                    "上一轮执行未成功。请分析失败原因，修复问题后继续完成原本的任务。\n"
                    "失败详情:\n" + result)}]
            result = self._run_messages(msgs, max_steps)
            if not result.startswith("[ERR]") and not result.startswith("[WARN]"):
                return result
        return (f"[ERR] 自动修复 {max_retries} 次后仍未成功，需要人工介入。\n"
                f"最后一次失败详情:\n{result}")


# 创建 agent 实例（默认 auto 模式）
agent = CodingAgent(api_key=os.environ["DEEPSEEK_API_KEY"])
console.print("[green]✅ Agent 初始化完成（bash: auto）[/]")

# ========================================
# 第4部分：基础功能演示
# ========================================

In [ ]:
# 示例1：纯对话（无需工具）
print("=== 示例1：基础对话 ===")
result = agent.run("用一句话介绍什么是 ReAct 循环。")
print(result)

In [ ]:
# 示例2：文件操作（write）
print("=== 示例2：创建文件 ===")
result = agent.run("创建一个 hello.py 文件，内容是一个打印 'Hello from wz-agent!' 的 Python 脚本")
print("\n最终结果:", result)

In [ ]:
# 验证文件是否真的创建了
!cat hello.py 2>/dev/null || type hello.py

In [ ]:
# 示例3：edit 精确修改（v0.2 新增能力）
print("=== 示例3：edit 修改文件 ===")
result = agent.run("用 read 读 hello.py，然后用 edit 把 'Hello from wz-agent!' 改成 'Hello v2.0 from wz-agent!'")
print("\n最终结果:", result)

In [ ]:
# 示例4：bash 执行命令 + plan 安全模式（v0.3+）
# 先看 auto 模式效果
print("=== 示例4a：auto 模式直接执行 ===")
result = agent.run("用 bash 运行 python hello.py 并告诉我输出结果")
print("\n最终结果:", result)

In [ ]:
# 示例4b：plan 模式 —— 命令先收集，确认后批量执行
print("=== 示例4b：plan 模式（先收集后执行）===")
agent_plan = CodingAgent(api_key=os.environ["DEEPSEEK_API_KEY"], bash_mode="plan")
result = agent_plan.run("列出当前目录文件（ls），然后运行 python hello.py。把这两个命令收集到计划后一次性执行")
print("\n最终结果:", result)

# ========================================
# 第5部分：多步骤任务演示
# ========================================

In [ ]:
# 示例5：多步骤任务 —— 创建一个猜数字游戏
print("=== 示例5：创建猜数字游戏 ===")
task = """
请帮我完成以下任务：
1. 创建一个 guess_number.py 文件，实现猜数字游戏（1-100 随机数，最多 7 次机会）
2. 运行 python guess_number.py 验证语法没问题
3. 用 read 工具读取 guess_number.py 内容，确认代码完整
"""
result = agent.run_with_retry(task)
print("\n最终结果:", result)

In [ ]:
# 验证生成的文件
!cat guess_number.py 2>/dev/null || type guess_number.py

# ========================================
# 第6部分：v2.0 —— triage + to-tickets
# ========================================

## 两个新能力

- **triage**：issue 分诊状态机，把 issue 移到五档标签之一：
  `needs-triage`（待评估）→ `needs-info`（缺信息，等人补充）/ `ready-for-agent`（描述完整可直接开工）/
  `ready-for-human`（需人处理）/ `wontfix`（不做）
- **to-tickets**：把 spec 拆成 **tracer-bullet 垂直切片** tickets ——
  每条切一个贯穿所有层的完整路径、可独立演示/验证；每个 ticket 声明被谁阻塞（`Blocked by`）

## 数据层：.scratch/ 本地 issue tracker

```
.scratch/<feature>/
├── spec.md              # 需求规格
└── issues/
    ├── 01-xxx.md        # 一个 ticket 一个文件，从 01 编号
    └── 02-yyy.md        # 顶部含 Status: / Type: / Blocked by: 行
```

状态机文件操作（编号唯一、标签合法、Status 行读写）由 `src/agent/issues.py` 负责——
**LLM 只做分析决策，写入的正确性由代码保证**。下面注册 3 个 v2 工具（薄封装委托真实实现）。

In [ ]:
# ---- v2.0 工具：迷你反射基类 + 委托真实实现 ----
# 对应 src/agent/tools/base.py 的反射原理：从 run() 签名自动生成 schema
import inspect
from pathlib import Path

# 把项目 src/ 加入路径（notebook 在项目根运行时）
sys.path.insert(0, str(Path.cwd() / "src"))
from agent import issues  # v2.0 数据层


class MiniBaseTool:
    """迷你反射基类：从 run() 签名自动生成 OpenAI function schema。"""

    TYPE_MAP = {int: "integer", float: "number", bool: "boolean", str: "string"}

    def to_openai_function(self) -> dict:
        sig = inspect.signature(self.run)
        props, required = {}, []
        for name, param in sig.parameters.items():
            if name == "self":
                continue
            jtype = self.TYPE_MAP.get(param.annotation, "string")
            props[name] = {"type": jtype, "description": f"{name} 参数"}
            if param.default is inspect.Parameter.empty:
                required.append(name)
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {"type": "object", "properties": props, "required": required},
            },
        }


class ListIssuesTool(MiniBaseTool):
    """列出 feature 下所有 issue 及当前状态。"""

    name = "list_issues"
    description = "列出指定 feature 下的所有 issue 文件及当前 Status"

    def run(self, feature: str) -> str:
        files = issues.list_issue_files(feature)
        if not files:
            return f"[INFO] feature '{feature}' 下没有 issue 文件"
        lines = [f"feature '{feature}' 共 {len(files)} 个 issue:"]
        for p in files:
            lines.append(f"  [{p.stem}] {p.name} —— Status: {issues.get_status(p)}")
        return "\n".join(lines)


class SetIssueStatusTool(MiniBaseTool):
    """更新 issue 的分诊状态（五标签状态机）。"""

    name = "set_issue_status"
    description = f"更新 issue 的分诊状态。合法标签: {', '.join(issues.VALID_LABELS)}"

    def run(self, feature: str, issue: str, label: str, comment: str = "") -> str:
        path = issues.issue_path(feature, issue)
        if path is None:
            return f"[ERR] 在 feature '{feature}' 中找不到 issue '{issue}'"
        return issues.set_status(path, label, comment)


class AllocateIssueTool(MiniBaseTool):
    """为 feature 分配下一个 issue 编号（写 ticket 前调用）。"""

    name = "allocate_issue"
    description = "为指定 feature 分配下一个可用的 issue 编号（编号唯一性由代码保证）"

    def run(self, feature: str) -> str:
        n = issues.next_issue_number(feature)
        return f"下一个可用编号: {n:02d} —— 请把 ticket 文件命名为 {n:02d}-<slug>.md"


# 注册到全局工具表（第2部分定义）
ALL_TOOLS.update({
    "list_issues": ListIssuesTool(),
    "set_issue_status": SetIssueStatusTool(),
    "allocate_issue": AllocateIssueTool(),
})
console.print("[green]✅ v2.0 工具已注册:[/] list_issues, set_issue_status, allocate_issue")

# ========================================
# 第7部分：v2.0 实战 —— to-tickets + triage
# ========================================

用教学版 Agent + 真实 prompt（`src/agent/prompts/`）+ 真实数据层跑一遍完整闭环：
**spec → to-tickets（拆成垂直切片）→ triage（分诊状态机）**。

In [ ]:
# ---- 准备演示 feature：一个待办 CLI 的 spec ----
demo = "demo-todo"
spec_file = issues.spec_path(demo)
spec_file.parent.mkdir(parents=True, exist_ok=True)
spec_file.write_text("""# 待办清单 CLI（演示用 spec）

## 功能需求
- 添加待办（add）、列出待办（list）、完成待办（done）
- 数据持久化到本地 todo.json

## 技术栈
- Python 3 标准库（无第三方依赖）

## 边界条件
- 数据文件损坏时给出友好错误提示

## 交付物
- todo.py + 冒烟测试脚本
""", encoding="utf-8")
console.print("[green]✅ 演示 spec 已就绪:[/]", str(spec_file))

In [ ]:
# ---- 实战1：to-tickets —— 把 spec 拆成垂直切片 tickets ----
from agent.prompts import TO_TICKETS_SYSTEM_PROMPT, TRIAGE_SYSTEM_PROMPT

print("=== to-tickets：任务拆解 ===")
result = agent.run_with_retry(
    TO_TICKETS_SYSTEM_PROMPT,
    f"请把以下 spec 拆解成垂直切片 tickets（写入 .scratch/{demo}/issues/）。\n"
    f"先 list_issues 确认现状，再逐个 allocate_issue + write。\n\n"
    f"===== spec =====\n" + spec_file.read_text(encoding="utf-8"),
    system_prompt=TO_TICKETS_SYSTEM_PROMPT,
)
print("\n最终结果:", result)

print("\n--- 生成的 tickets ---")
for p in issues.list_issue_files(demo):
    print(f"\n📄 {p.name}\n" + p.read_text(encoding="utf-8"))

In [ ]:
# ---- 实战2：triage —— 分诊生成的 tickets（状态机）----
print("=== triage：issue 分诊 ===")
result = agent.run_with_retry(
    TRIAGE_SYSTEM_PROMPT,
    f"请分诊 feature '{demo}' 下的所有 issue。",
    system_prompt=TRIAGE_SYSTEM_PROMPT,
)
print("\n最终结果:", result)

print("\n--- 分诊后的状态 ---")
for p in issues.list_issue_files(demo):
    print(f"  {p.name} —— Status: {issues.get_status(p)}")

In [ ]:
# ---- 实战3：再看一个 issue 文件内部长什么样 ----
print("第一个 ticket 的内容：")
first = issues.list_issue_files(demo)[0]
print(first.read_text(encoding="utf-8"))

# ========================================
# 第8部分：清理临时文件
# ========================================

In [ ]:
# 清理演示过程中创建的文件
import shutil
for f in ["hello.py", "guess_number.py"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"🗑️ 已删除：{f}")

if os.path.exists(".scratch"):
    shutil.rmtree(".scratch")
    print("🗑️ 已删除 .scratch/（演示产生的 tickets）")
print("✅ 清理完成")

# ========================================
# 第9部分：总结
# ========================================

## 本教程覆盖（v2.0 版）

| 能力 | 对应版本 | 本教程位置 |
|------|---------|-----------|
| DeepSeek API 连通（OpenAI 兼容） | v0.1 | 第1部分 |
| ReAct 循环 + 工具调用 | v0.2 | 第3、4部分 |
| edit 工具（精确替换 + 唯一性校验） | v0.2 | 第2、4部分 |
| bash 安全模式（auto / plan 双模式） | v0.3 | 第2、4部分 |
| 自动重试（失败回喂 LLM，最多 3 次） | v1.0 | 第3部分 |
| triage（issue 分诊状态机） | v2.0 | 第6、7部分 |
| to-tickets（垂直切片任务拆解） | v2.0 | 第6、7部分 |

## 项目里程碑

- ✅ **v0.1 ~ v0.2**：API 连通、ReAct 循环、read/write/edit/bash 四工具
- ✅ **v0.3 ~ v1.0**：bash 安全模式、两阶段（需求澄清 → 编码执行）、click CLI、自动重试
- ✅ **v2.0**：triage + to-tickets 集成

## 教学版 vs 完整实现（src/agent/）

本教程的 `CodingAgent` 是**简化教学版**（约 200 行，用于讲解原理）；
`src/agent/` 里的完整实现在此基础上增加了：

- **schema 全反射**：`BaseTool.to_openai_function()` 用 inspect 自动生成，无需手写
- **工具注册表**：`ALL_TOOLS` 字典 + 名字冲突即报错
- **bash 安全模式**：危险命令前缀拦截 + 计划模式批量执行（30s 超时）
- **自动重试**：`run_with_retry()` 消息复用 + [ERR]/[WARN] 判定
- **数据层**：`.scratch/` issue 操作层（`issues.py`）+ 三个 v2 专用工具
- **CLI**：click 子命令分发（`python src/main.py triage|to-tickets <目标>`）

> 从零手搓的核心价值：ReAct 循环不到 100 行、无框架依赖，
> 每一步都可以被理解、被修改、被扩展。